In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio import mask
import geopandas as gpd
import math
#from osgeo import gdal

PLOT_ID_INDEX = 0

# File paths (edit before running)
rgb_tif_path = 'path/to/RGB/YYYY-MM-DD/rgb_orthomosaic.tif'
shp_path = 'path/to/RGB/YYYY-MM-DD/plot_grid.shp'
out_csv_path = 'path/to/output/rgb_vi.csv'

# Open the raster data
src = rasterio.open(rgb_tif_path)
num_bands = src.count
print("num bands:", num_bands)
# Read all the bands
# Not used; commented out to avoid loading the full orthomosaic into memory.
# img = src.read()
# Open the shapefile data
plots = gpd.read_file(shp_path)
header = plots.dtypes
print("Header\n", header)
print(plots.iloc[1, -1])
# src.nodata
# Not used; commented out to avoid loading the full orthomosaic into memory.
# img = img.astype(np.float32)
# # set no data value
# img[img == np.float32(src.nodata)] = np.nan
# #  (----------condition----------)   (-value-)
# img.shape
# import re
def extract_by_mask(src, shape_geom): # shape_geom = plots.iloc[i, -1] :: -1 == geometry
    """
    Extract the masked image based on the given shape geometry.
    Arguments:
      - src: a rasterio image object
      - shape_geom: the shapely geometry object
    Returns:
      - a cropped image
    """
    # Apply the masking 
    mask_img = mask.mask(dataset=src, shapes=[shape_geom], nodata=0, filled=True, crop=True)[0]
    # print("mask_img.shape:\n",mask_img.shape)
    # Get the mean values
    red_mean   = np.nanmean(mask_img[0, :, :])
    green_mean = np.nanmean(mask_img[1, :, :])
    blue_mean  = np.nanmean(mask_img[2, :, :])
    #RCC
    rcc = red_mean / (red_mean + green_mean + blue_mean)
    #GCC
    gcc =  green_mean / (red_mean + green_mean + blue_mean)
    #ExR
    exr = (1.4 * red_mean - green_mean) / (red_mean + green_mean + blue_mean)
    # exg
    exg = (2 * green_mean - red_mean - blue_mean) / (green_mean + red_mean + blue_mean)
    # ExGR
    exgr = exg - exr
    # VARI_2
    vari_2 = (green_mean - red_mean) / (green_mean + red_mean - blue_mean)
    # MGRVI
    mgrvi = (np.square (green_mean) - np.square (red_mean)) / (np.square (green_mean) + np.square (red_mean))
    # CIVE
    cive = (0.441 * red_mean) - (0.881 * green_mean) + (0.385 * blue_mean) + 18.787
    # VEG 
    veg = green_mean /(np.power (red_mean,0.667) * np.power(blue_mean,0.334))
    # rgvi
    rgvi = red_mean / green_mean
    # npci
    npci = (red_mean - blue_mean) / (red_mean + blue_mean)
    # srpi
    srpi = blue_mean / red_mean
    # ngrdi (Normalized green red difference index)
    ngrdi = (green_mean - red_mean) / (green_mean + red_mean)
    #sci (Soil Colour Index)
    sci = (red_mean - green_mean) / (red_mean + green_mean)
    #gli (Green leaf index )
    gli = (2 * green_mean - red_mean - blue_mean) / (2 * green_mean + red_mean + blue_mean)
    #hi (Primary colours Hue Index)
    hi = (2 * red_mean - green_mean  - blue_mean) / (green_mean  - blue_mean)
    #vari (A Visible Atmospherically Resistant Index)
    vari = (green_mean - red_mean) / (green_mean + red_mean + blue_mean)
    #glai
    glai = (25 * (green_mean - red_mean) / (green_mean + red_mean - blue_mean) + 1.25)
    #ci (Coloration Index)
    ci = (red_mean - blue_mean) / (red_mean)
    #shp (Shape index)
    shp = 2 * (red_mean - green_mean  - blue_mean) / (green_mean  - blue_mean)
    #sat (Overall Saturation Index)
    sat = (max([red_mean,green_mean,blue_mean]) - min([red_mean,green_mean,blue_mean])) / max([red_mean,green_mean,blue_mean])
    #hue (Overall Hue Index)
    hue = math.atan(2 * (red_mean - green_mean  - blue_mean) / 30.5 * (green_mean  - blue_mean))
    #ri (Redness Index)
    ri = np.square (red_mean) / ((blue_mean) * np.power(green_mean, 3))
    #bi brightness index
    bi = math.sqrt((np.square(red_mean) + np.square (green_mean) + np.square (blue_mean))/3)
    #vvi (visible vegetation index)
    vvi = (1 - (red_mean - 30) / (red_mean + 30)) * (1 - (green_mean - 50) / (green_mean + 50)) * (1 - (blue_mean - 1)/(blue_mean + 1))

    return (blue_mean, green_mean, red_mean, rcc, gcc, exr, exg, exgr, vari_2, mgrvi, cive, veg, rgvi, npci, srpi, ngrdi, sci, gli, hi, 
            vari, glai, ci, shp, sat, hue, ri, bi, vvi)
# Apply this function as for loop for all features
# Get the total number of features
num_features = plots.shape[0]
# Create empty lists to hold the information 
list_plots = []
list_blue = []
list_green = []
list_red = []
list_rcc = []
list_gcc = []
list_exr = []
list_exg = []
list_exgr = []
list_vari_2 = []
list_mgrvi = []
list_cive = []
list_veg = []
list_rgvi = []
list_npci = []
list_srpi = []
list_ngrdi = []
list_sci = []
list_gli = []
list_hi = []
list_vari = []
list_glai = []
list_ci = []
list_shp = []
list_sat = []
list_hue = []
list_ri = []
list_bi = [] 
list_vvi = []
# Iterate through that range of numbers
for i in range(num_features):
    # save geometry information 
    shape_geom = plots.iloc[i, -1]
    # save plotid information 
    plotid = plots.iloc[i, PLOT_ID_INDEX]
    # print(plotid)
    # Apply mask function
    (blue_mean, green_mean, red_mean, rcc, gcc, exr, exg, exgr, vari_2, mgrvi, cive, veg, rgvi, npci, srpi, ngrdi, sci, gli, hi, 
            vari, glai, ci, shp, sat, hue, ri, bi, vvi) = extract_by_mask(src, shape_geom)
     
    # Append the information to each empty list
    list_plots.append(plotid)
    list_blue.append(blue_mean)
    list_green.append(green_mean)
    list_red.append(red_mean)
    list_rcc.append(rcc)
    list_gcc.append(gcc)
    list_exr.append(exr)
    list_exg.append(exg)
    list_exgr.append(exgr)
    list_vari_2.append(vari_2)
    list_mgrvi.append(mgrvi)
    list_cive.append(cive)
    list_veg.append(veg)
    list_rgvi.append(rgvi)
    list_npci.append(npci)
    list_srpi.append(srpi)
    list_ngrdi.append(ngrdi)
    list_sci.append(sci)
    list_gli.append(gli)
    list_hi.append(hi)
    list_vari.append(vari)
    list_glai.append(glai)
    list_ci.append(ci)
    list_shp.append(shp)
    list_sat.append(sat)
    list_hue.append(hue)
    list_ri.append(ri)
    list_bi.append(bi)
    list_vvi.append(vvi)
# Create the dataframe
mean_indicies = pd.DataFrame({
    'var': list_plots,
    'Mean Blue': list_blue,
    'Mean Green': list_green,
    'Mean Red': list_red,
    'RCC': list_rcc,
    'GCC': list_gcc,
    'ExR': list_exr,
    'ExG': list_exg,
    'ExGR': list_exgr,
    'VARI_2': list_vari_2,
    'MGRVI': list_mgrvi,
    'CIVE': list_cive,
    'VEG': list_veg,
    'RGVI' : list_rgvi,
    'NPCI' : list_npci,
    'SRPI' : list_srpi,
    'NGRDI' : list_ngrdi,
    'SCI' : list_sci,
    'GLI' : list_gli,
    'HI' : list_hi,
    'VARI' : list_vari,
    'GLAI' : list_glai,
    'CI' : list_ci,
    'SHP' : list_shp,
    'SAT' : list_sat,
    'HUE' : list_hue,
    'RI' : list_ri,
    'BI' : list_bi,
    'VVI' : list_vvi,
})
mean_indicies.head()
# add spectral indices info to original plots df based Plot_ID
df = pd.concat(
    [plots.reset_index(drop=True), mean_indicies.drop(columns=['var']).reset_index(drop=True)],
    axis=1
)
# df
# export csv to drive
# df.to_csv ('./20200904_Bradford_H1G_MS_VIs.csv', index = None, header=True)
df.to_csv (out_csv_path, index = None, header=True)